# Preprocessing for classification

Notebook này chuẩn bị dữ liệu cho bài toán phân loại tái nhập viện từ file clean chung `data/diabetic_data_clean_common.csv`.

Mục tiêu:

- Chọn biến mục tiêu `readmitted_binary` cho phân loại nhị phân.
- Bỏ các cột không dùng trực tiếp trong mô hình phân loại.
- Chia train/validation/test trước khi fit encoder/scaler để tránh data leakage.
- Mã hóa biến phân loại bằng one-hot encoding.
- Scale biến số bằng standard scaling.
- Lưu các file train/validation/test đã xử lý để dùng cho bước modeling.


In [1]:
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    from sklearn.compose import ColumnTransformer
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
except ImportError as exc:
    raise ImportError(
        "Notebook này cần pandas, numpy và scikit-learn. Hãy cài bằng: pip install pandas numpy scikit-learn"
    ) from exc

pd.set_option("display.max_columns", 120)


In [2]:
cwd = Path.cwd().resolve()
ROOT_DIR = cwd if (cwd / "data" / "diabetic_data_clean_common.csv").exists() else cwd.parent

INPUT_PATH = ROOT_DIR / "data" / "diabetic_data_clean_common.csv"
OUTPUT_DIR = ROOT_DIR / "data" / "classification"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH, OUTPUT_DIR


(WindowsPath('D:/HocTap/KT&XLTT/CUOIKI/data/diabetic_data_clean_common.csv'),
 WindowsPath('D:/HocTap/KT&XLTT/CUOIKI/data/classification'))

## 1. Load dữ liệu clean chung


In [3]:
df = pd.read_csv(INPUT_PATH)

print("Shape:", df.shape)
df.head()


Shape: (101763, 51)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_binary,age_midpoint,age_ordinal,diag_1_group,diag_2_group,diag_3_group
0,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,NaN,NaN,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO,0,5,0,Endocrine_Metabolic,Unknown,Unknown
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30,1,15,1,Endocrine_Metabolic,Endocrine_Metabolic,Endocrine_Metabolic
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO,0,25,2,Pregnancy_Childbirth,Endocrine_Metabolic,Supplementary_V
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO,0,35,3,Infectious_Parasitic,Endocrine_Metabolic,Circulatory
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO,0,45,4,Neoplasms,Neoplasms,Endocrine_Metabolic


## 2. Chọn target và feature

Ở notebook này dùng `readmitted_binary` làm target:

- `0`: không tái nhập viện (`NO`)
- `1`: có tái nhập viện (`<30` hoặc `>30`)

Các cột bị bỏ khỏi feature:

- `readmitted`, `readmitted_binary`: target hoặc target gốc.
- `diag_1`, `diag_2`, `diag_3`: mã ICD-9 gốc có quá nhiều giá trị; dùng `diag_*_group` thay thế.
- `age`, `age_midpoint`: giữ `age_ordinal` để biểu diễn thứ tự nhóm tuổi và tránh trùng thông tin.


In [4]:
TARGET_COL = "readmitted_binary"

drop_from_features = [
    "readmitted",
    "readmitted_binary",
    "diag_1",
    "diag_2",
    "diag_3",
    "age",
    "age_midpoint",
]

X = df.drop(columns=[col for col in drop_from_features if col in df.columns])
y = df[TARGET_COL].astype(int)

print("X shape:", X.shape)
print("y distribution:")
display(y.value_counts(normalize=True).rename("rate").to_frame())
X.head()
y.head(5)

X shape: (101763, 44)
y distribution:


,rate
readmitted_binary,
0,0.539106
1,0.460894


0    0
1    1
2    0
3    0
4    0
Name: readmitted_binary, dtype: int64

## 3. Chia train/validation/test

Chia dữ liệu trước khi fit preprocessing để encoder và scaler chỉ học từ tập train.

Tỷ lệ sử dụng trong notebook này:

- Train: 64%
- Validation: 16%
- Test: 20%


In [5]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    random_state=42,
    stratify=y_train_val,
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("Train target rate:")
display(y_train.value_counts(normalize=True).rename("rate").to_frame())
print("Validation target rate:")
display(y_val.value_counts(normalize=True).rename("rate").to_frame())
print("Test target rate:")
display(y_test.value_counts(normalize=True).rename("rate").to_frame())


X_train: (65128, 44)
X_val: (16282, 44)
X_test: (20353, 44)
Train target rate:


,rate
readmitted_binary,
0,0.539108
1,0.460892


Validation target rate:


,rate
readmitted_binary,
0,0.539123
1,0.460877


Test target rate:


,rate
readmitted_binary,
0,0.539085
1,0.460915


## 4. Xác định nhóm cột số và cột phân loại


In [6]:
categorical_id_features = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]
categorical_id_features = [col for col in categorical_id_features if col in X_train.columns]

numeric_features = [
    col for col in X_train.select_dtypes(include=["number"]).columns.tolist()
    if col not in categorical_id_features
]
categorical_features = (
    X_train.select_dtypes(exclude=["number"]).columns.tolist()
    + categorical_id_features
)

print("Numeric features:", len(numeric_features))
print(numeric_features)
print("\nCategorical features:", len(categorical_features))
print(categorical_features)


Numeric features: 9
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'age_ordinal']

Categorical features: 35
['race', 'gender', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']


## 5. Tạo preprocessing pipeline

- Biến số: scale bằng `StandardScaler`.
- Biến phân loại: one-hot encode, bỏ qua category mới nếu xuất hiện ở test.


In [7]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop",
)

preprocessor


,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,copy,True
,with_mean,True
,with_std,True


## 6. Fit trên train và transform train/validation/test


In [8]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_val_processed = pd.DataFrame(X_val_processed, columns=feature_names, index=X_val.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print("Processed X_train:", X_train_processed.shape)
print("Processed X_val:", X_val_processed.shape)
print("Processed X_test:", X_test_processed.shape)
X_train_processed.head()


Processed X_train: (65128, 291)
Processed X_val: (16282, 291)
Processed X_test: (20353, 291)


,num__time_in_hospital,num__num_lab_procedures,num__num_procedures,num__num_medications,num__number_outpatient,num__number_emergency,num__number_inpatient,num__number_diagnoses,num__age_ordinal,cat__race_AfricanAmerican,cat__race_Asian,cat__race_Caucasian,cat__race_Hispanic,cat__race_Other,cat__race_Unknown,cat__gender_Female,cat__gender_Male,cat__payer_code_BC,cat__payer_code_CH,cat__payer_code_CM,cat__payer_code_CP,cat__payer_code_DM,cat__payer_code_HM,cat__payer_code_MC,cat__payer_code_MD,cat__payer_code_MP,cat__payer_code_OG,cat__payer_code_OT,cat__payer_code_PO,cat__payer_code_SI,cat__payer_code_SP,cat__payer_code_UN,cat__payer_code_Unknown,cat__payer_code_WC,cat__medical_specialty_AllergyandImmunology,cat__medical_specialty_Anesthesiology,cat__medical_specialty_Anesthesiology-Pediatric,cat__medical_specialty_Cardiology,cat__medical_specialty_Cardiology-Pediatric,cat__medical_specialty_DCPTEAM,cat__medical_specialty_Dentistry,cat__medical_specialty_Dermatology,cat__medical_specialty_Emergency/Trauma,cat__medical_specialty_Endocrinology,cat__medical_specialty_Endocrinology-Metabolism,cat__medical_specialty_Family/GeneralPractice,cat__medical_specialty_Gastroenterology,cat__medical_specialty_Gynecology,cat__medical_specialty_Hematology,cat__medical_specialty_Hematology/Oncology,cat__medical_specialty_Hospitalist,cat__medical_specialty_InfectiousDiseases,cat__medical_specialty_InternalMedicine,cat__medical_specialty_Nephrology,cat__medical_specialty_Neurology,cat__medical_specialty_Neurophysiology,cat__medical_specialty_Obsterics&Gynecology-GynecologicOnco,cat__medical_specialty_Obstetrics,cat__medical_specialty_ObstetricsandGynecology,cat__medical_specialty_Oncology,...,cat__diag_3_group_Neoplasms,cat__diag_3_group_Nervous_Sense_Organs,cat__diag_3_group_Pregnancy_Childbirth,cat__diag_3_group_Respiratory,cat__diag_3_group_Skin,cat__diag_3_group_Supplementary_E,cat__diag_3_group_Supplementary_V,cat__diag_3_group_Symptoms,cat__diag_3_group_Unknown,cat__admission_type_id_1,cat__admission_type_id_2,cat__admission_type_id_3,cat__admission_type_id_4,cat__admission_type_id_5,cat__admission_type_id_6,cat__admission_type_id_7,cat__admission_type_id_8,cat__discharge_disposition_id_1,cat__discharge_disposition_id_2,cat__discharge_disposition_id_3,cat__discharge_disposition_id_4,cat__discharge_disposition_id_5,cat__discharge_disposition_id_6,cat__discharge_disposition_id_7,cat__discharge_disposition_id_8,cat__discharge_disposition_id_9,cat__discharge_disposition_id_10,cat__discharge_disposition_id_11,cat__discharge_disposition_id_12,cat__discharge_disposition_id_13,cat__discharge_disposition_id_14,cat__discharge_disposition_id_15,cat__discharge_disposition_id_16,cat__discharge_disposition_id_17,cat__discharge_disposition_id_18,cat__discharge_disposition_id_19,cat__discharge_disposition_id_20,cat__discharge_disposition_id_22,cat__discharge_disposition_id_23,cat__discharge_disposition_id_24,cat__discharge_disposition_id_25,cat__discharge_disposition_id_27,cat__discharge_disposition_id_28,cat__admission_source_id_1,cat__admission_source_id_2,cat__admission_source_id_3,cat__admission_source_id_4,cat__admission_source_id_5,cat__admission_source_id_6,cat__admission_source_id_7,cat__admission_source_id_8,cat__admission_source_id_9,cat__admission_source_id_10,cat__admission_source_id_11,cat__admission_source_id_13,cat__admission_source_id_14,cat__admission_source_id_17,cat__admission_source_id_20,cat__admission_source_id_22,cat__admission_source_id_25
67415,-1.136355,-1.068716,-0.198435,-0.737318,-0.294351,-0.214072,-0.502511,-1.764091,-1.934411,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.

## 7. Gộp target và lưu output


In [9]:
train_processed = X_train_processed.copy()
train_processed[TARGET_COL] = y_train.values

val_processed = X_val_processed.copy()
val_processed[TARGET_COL] = y_val.values

test_processed = X_test_processed.copy()
test_processed[TARGET_COL] = y_test.values

train_path = OUTPUT_DIR / "classification_train_64_processed.csv"
val_path = OUTPUT_DIR / "classification_validation_16_processed.csv"
test_path = OUTPUT_DIR / "classification_test_20_processed.csv"
feature_names_path = OUTPUT_DIR / "classification_feature_names_3split.csv"

train_processed.to_csv(train_path, index=False)
val_processed.to_csv(val_path, index=False)
test_processed.to_csv(test_path, index=False)
pd.Series(feature_names, name="feature_name").to_csv(feature_names_path, index=False)

print(f"Saved train data to: {train_path}")
print(f"Saved validation data to: {val_path}")
print(f"Saved test data to: {test_path}")
print(f"Saved feature names to: {feature_names_path}")


Saved train data to: D:\HocTap\KT&XLTT\CUOIKI\data\classification\classification_train_64_processed.csv
Saved validation data to: D:\HocTap\KT&XLTT\CUOIKI\data\classification\classification_validation_16_processed.csv
Saved test data to: D:\HocTap\KT&XLTT\CUOIKI\data\classification\classification_test_20_processed.csv
Saved feature names to: D:\HocTap\KT&XLTT\CUOIKI\data\classification\classification_feature_names_3split.csv


## 8. Kiểm tra nhanh output


In [10]:
print("Train processed shape:", train_processed.shape)
print("Validation processed shape:", val_processed.shape)
print("Test processed shape:", test_processed.shape)
print("Missing in train:", train_processed.isna().sum().sum())
print("Missing in validation:", val_processed.isna().sum().sum())
print("Missing in test:", test_processed.isna().sum().sum())

train_processed[[TARGET_COL]].value_counts(normalize=True).rename("rate").to_frame()


Train processed shape: (65128, 292)
Validation processed shape: (16282, 292)
Test processed shape: (20353, 292)
Missing in train: 0
Missing in validation: 0
Missing in test: 0


,rate
readmitted_binary,
0,0.539108
1,0.460892
